# Handwritten digits (MNIST) with an ANN in Keras
Multi-class classification: which digit, 0 to 9, is in a 28 x 28 image? Flatten, one hidden ReLU layer, a softmax output of 10 nodes.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # hide TensorFlow's log messages
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"  # take GPU memory only as needed
import json
from pathlib import Path

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import accuracy_score, confusion_matrix

# same random numbers on every run
keras.utils.set_random_seed(1)
tf.config.experimental.enable_op_determinism()

## The data
Keras ships MNIST: 60,000 training and 10,000 test images, already split.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

In [ ]:
# one image is a 28 x 28 array of pixel values from 0 to 255
print(X_train[0].shape, X_train[0].min(), X_train[0].max())
print(y_train[:3])

In [ ]:
# the first training image as a picture (it shows a 5)
px.imshow(X_train[0], color_continuous_scale="gray_r").show()

In [ ]:
# scale every pixel into 0..1
X_train = X_train / 255
X_test = X_test / 255
print(X_train[0].min(), X_train[0].max())

## First network: Flatten -> 128 ReLU -> 10 softmax

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28)),             # one image
    layers.Flatten(),                         # 28 x 28 -> 784
    layers.Dense(128, activation="relu"),     # hidden layer
    layers.Dense(10, activation="softmax"),   # one node per digit
])
model.summary()

In [ ]:
# labels are plain integers 0..9, so the "sparse" version of the loss
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
history1 = model.fit(X_train, y_train, epochs=10, validation_split=0.2, verbose=2)

In [ ]:
# 10 probabilities per test image
y_prob = model.predict(X_test, verbose=0)
print(y_prob.shape)
print(y_prob[0].round(3), y_prob[0].sum())

In [ ]:
# the predicted digit is the position of the largest probability
y_pred = y_prob.argmax(axis=1)
print(y_pred[:3], y_test[:3])
acc1 = accuracy_score(y_test, y_pred)
print("accuracy:", acc1)

## The other loss: categorical cross-entropy needs one-hot labels

In [ ]:
print(y_train[0], keras.utils.to_categorical(y_train[0], 10))

## Second network: an extra hidden layer of 32 nodes, 25 epochs, accuracy tracked

In [ ]:
model2 = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
model2.summary()

In [ ]:
model2.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
history = model2.fit(X_train, y_train, epochs=25, validation_split=0.2, verbose=2)

In [ ]:
y_pred2 = model2.predict(X_test, verbose=0).argmax(axis=1)
acc2 = accuracy_score(y_test, y_pred2)
cm = confusion_matrix(y_test, y_pred2)
print("accuracy:", acc2)
print(cm)

In [ ]:
h = history.history
best = int(np.argmin(h["val_loss"]))
print("lowest val_loss at epoch", best + 1, round(h["val_loss"][best], 4))
print({k: round(v[-1], 4) for k, v in h.items()})

## Training curves

In [ ]:
epochs = list(range(1, len(h["loss"]) + 1))
for key in ["loss", "accuracy"]:
    fig = go.Figure()
    fig.add_scatter(x=epochs, y=h[key], name="training " + key)
    fig.add_scatter(x=epochs, y=h["val_" + key], name="validation " + key)
    fig.update_layout(template="simple_white", xaxis_title="epoch", yaxis_title=key)
    fig.show()

## Predicting one image

In [ ]:
# predict expects a batch, so one image is reshaped to (1, 28, 28)
for i in [0, 1]:
    p = model2.predict(X_test[i].reshape(1, 28, 28), verbose=0)
    print("image", i, "label", y_test[i], "predicted", p.argmax(axis=1)[0], "probability", p.max().round(4))

In [ ]:
# the test images the second network gets wrong, and what it says instead
wrong = np.where(y_pred2 != y_test)[0]
print(len(wrong), "wrong;", "first few:", [(int(i), int(y_test[i]), int(y_pred2[i])) for i in wrong[:6]])

In [ ]:
# save the numbers the Note's figures are drawn from (images/figs.py)
results = {"history1": history1.history, "history": h, "cm": cm.tolist(), "acc1": acc1, "acc2": acc2,
           "train_images": (X_train[:10] * 255).round().astype(int).tolist(), "train_labels": y_train[:10].tolist(),
           "wrong_images": (X_test[wrong[:5]] * 255).round().astype(int).tolist(),
           "wrong_true": y_test[wrong[:5]].tolist(), "wrong_pred": y_pred2[wrong[:5]].tolist()}
Path("images/results.json").write_text(json.dumps(results))